In [6]:
from langchain_openai import ChatOpenAI
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel

In [2]:
load_dotenv()
model_1 = ChatOpenAI()

In [3]:
llm = HuggingFaceEndpoint(
    repo_id="deepseek-ai/DeepSeek-V3-0324:fastest",
    task="text-generation"
)

model_2 = ChatHuggingFace(llm=llm)

/Users/harshsharma/Documents/Python/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
prompt_1 = PromptTemplate(
    template="Generate short and simple notes from the following text \n {text}",
    input_variables=['text']
)

prompt_2 = PromptTemplate(
    template="Generate 5 short questions from the follwing text \n {text}",
    input_variables=['text']
)

prompt_3 = PromptTemplate(
    template="Merge the given notes and quiz into a single document \n notes -> {notes}, quiz -> {quiz}",
    input_variables=['notes', 'quiz']
)

In [5]:
parser = StrOutputParser()

In [7]:
parallel_chain = RunnableParallel({
    'notes': prompt_1 | model_1 | parser,
    'quiz': prompt_2 | model_2 | parser
})

merge_chain = prompt_3 | model_1 | parser

chain = parallel_chain | merge_chain

In [8]:
text = """
Support vector machines (SVMs) are a set of supervised learning methods used for classification, regression and outliers detection.

The advantages of support vector machines are:

Effective in high dimensional spaces.

Still effective in cases where number of dimensions is greater than the number of samples.

Uses a subset of training points in the decision function (called support vectors), so it is also memory efficient.

Versatile: different Kernel functions can be specified for the decision function. Common kernels are provided, but it is also possible to specify custom kernels.

The disadvantages of support vector machines include:

If the number of features is much greater than the number of samples, avoid over-fitting in choosing Kernel functions and regularization term is crucial.

SVMs do not directly provide probability estimates, these are calculated using an expensive five-fold cross-validation (see Scores and probabilities, below).

The support vector machines in scikit-learn support both dense (numpy.ndarray and convertible to that by numpy.asarray) and sparse (any scipy.sparse) sample vectors as input. However, to use an SVM to make predictions for sparse data, it must have been fit on such data. For optimal performance, use C-ordered numpy.ndarray (dense) or scipy.sparse.csr_matrix (sparse) with dtype=float64.
"""

result = chain.invoke({'text': text})
print(result)

Support Vector Machines (SVMs):
- Used for classification, regression, and outliers detection
- Advantages: effective in high dimensions, memory efficient, versatile with kernel functions
- Disadvantages: potential over-fitting, do not provide direct probability estimates
- Scikit-learn SVMs support both dense and sparse sample vectors for input

Quiz:
1. What are the main applications of Support Vector Machines (SVMs)?
2. Why are SVMs considered memory efficient?
3. What is a key advantage of SVMs in high-dimensional spaces?
4. What is a major disadvantage of SVMs regarding probability estimates?
5. What input data types are supported by scikit-learn's SVM implementation?

Let me know if you need any further assistance or clarification!


In [9]:
chain.get_graph().print_ascii()

            +---------------------------+            
            | Parallel<notes,quiz>Input |            
            +---------------------------+            
                 **               **                 
              ***                   ***              
            **                         **            
+----------------+                +----------------+ 
| PromptTemplate |                | PromptTemplate | 
+----------------+                +----------------+ 
          *                               *          
          *                               *          
          *                               *          
  +------------+                 +-----------------+ 
  | ChatOpenAI |                 | ChatHuggingFace | 
  +------------+                 +-----------------+ 
          *                               *          
          *                               *          
          *                               *          
+-----------------+         